# Moon & Lazar (2023), Figure 5 — full-fidelity reproduction

Reproduces **both panels of Figure 5** at the paper's own budget of 500 replications, rather than the 150 the pytest version uses to stay fast locally.

| curve | what it is |
|---|---|
| Fig. 5a | false-positive rate of the two-stage test vs noise level |
| Fig. 5b, "PI" | power of the two-stage test vs noise level |
| Fig. 5b, "PD" | the same design run through Robinson & Turner (2017) — an independent published curve for our `rt` wrapper |

**Design** (paper Section 4.1, itself Robinson & Turner's): shape 1 is one circle of radius 1, shape 2 is two circles of radii 0.9 and 1.1; 50 points per cloud plus `N(0, sigma^2)` noise; Rips complex; dimension-one diagrams. The false-positive scenario draws 20 clouds from shape 2 and splits them at random into two groups of 10; the power scenario draws 10 from each shape.

**Cost**: ~10–15 min on 32 cores, peak RSS well under 4 GB. Workers are forked, so the imported libraries are shared copy-on-write and each worker adds only a few MB.

**Sharding**: replication `r` is seeded from `(base_seed, r)` alone, so shards concatenate into exactly the result a single sequential process would produce. `N_JOBS` and `CHUNK` change the wall clock, never the numbers.

In [ ]:
# Must run BEFORE numpy is imported anywhere: without this each of the N_JOBS
# worker processes spins up its own BLAS thread pool and oversubscribes the box.
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

print("logical CPUs:", os.cpu_count())
!free -g | head -2

In [ ]:
REPO_URL = "https://github.com/hugogobato/Pointcloud_Equality_Testing.git"
REPO_DIR = "Pointcloud_Equality_Testing"

import os

if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL
else:
    !cd $REPO_DIR && git pull -q

# Only the PH + competitor path is needed here, so install those directly and
# then the package itself with --no-deps. This deliberately keeps Colab's own
# numpy/scipy instead of the repo's pins: the pinned numpy would force a
# runtime restart, and nothing in this notebook depends on that exact version.
!pip install -q gudhi==3.11.0 ripser==0.6.10 persim==0.3.8
!pip install -q --no-deps -e $REPO_DIR

import sys

if os.path.abspath(REPO_DIR) not in sys.path:
    sys.path.insert(0, os.path.abspath(REPO_DIR))
print("ready")

In [ ]:
import json
import time

import numpy as np

from tda2s.repro import MOON_LAZAR_FIG5, MOON_LAZAR_FIG5B_RT, MOON_LAZAR_SETTINGS
from tda2s.repro.parallel import default_workers, run_moon_lazar_grid

REPS = 500                      # the paper's budget
SIGMAS = (0.05, 0.10, 0.15, 0.20)
SCENARIOS = ("fpr", "power")
N_JOBS = default_workers(cap=32)  # raise the cap if you have RAM headroom
CHUNK = 25                      # replications per shard

print(f"{REPS} reps x {len(SIGMAS)} sigmas x {len(SCENARIOS)} scenarios "
      f"on {N_JOBS} workers")
print("two-stage settings:", MOON_LAZAR_SETTINGS)

In [ ]:
# ---- Figure 5a + 5b "PI": the two-stage persistence-image test ----------
t0 = time.perf_counter()
pi_results = run_moon_lazar_grid(
    "moon_lazar", SIGMAS, SCENARIOS, REPS,
    base_seed=0, n_jobs=N_JOBS, chunk=CHUNK, **MOON_LAZAR_SETTINGS)
print(f"\nmoon_lazar grid: {time.perf_counter() - t0:.1f}s")

In [ ]:
# ---- Figure 5b "PD": the Robinson-Turner curve on the same design ------
# Slower per replication (pairwise Wasserstein inside every permutation), so
# it gets a smaller chunk to keep the workers evenly fed.
t0 = time.perf_counter()
pd_results = run_moon_lazar_grid(
    "rt", SIGMAS, SCENARIOS, REPS,
    base_seed=7000, n_jobs=N_JOBS, chunk=10,
    metric="wasserstein", statistic="within", n_perm=200, seed=0)
print(f"\nrt grid: {time.perf_counter() - t0:.1f}s")

In [ ]:
rows = []
for sigma in SIGMAS:
    ref = MOON_LAZAR_FIG5[sigma]
    rows.append({
        "sigma": sigma,
        "pi_fpr": pi_results[(sigma, "fpr")]["rate"],
        "pi_fpr_se": pi_results[(sigma, "fpr")]["se"],
        "pi_fpr_published": ref["fpr"],
        "pi_power": pi_results[(sigma, "power")]["rate"],
        "pi_power_se": pi_results[(sigma, "power")]["se"],
        "pi_power_published": ref["power"],
        "pd_power": pd_results[(sigma, "power")]["rate"],
        "pd_power_se": pd_results[(sigma, "power")]["se"],
        "pd_power_published": MOON_LAZAR_FIG5B_RT[sigma],
        "pd_fpr": pd_results[(sigma, "fpr")]["rate"],
    })

hdr = f"{'sigma':>6} | {'FPR (pub)':>16} | {'PI power (pub)':>20} | {'PD power (pub)':>20}"
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(f"{r['sigma']:6.2f} | "
          f"{r['pi_fpr']:.3f}+-{r['pi_fpr_se']:.3f} ({r['pi_fpr_published']:.3f}) | "
          f"{r['pi_power']:.3f}+-{r['pi_power_se']:.3f} ({r['pi_power_published']:.2f})     | "
          f"{r['pd_power']:.3f}+-{r['pd_power_se']:.3f} ({r['pd_power_published']:.2f})")

os.makedirs("results", exist_ok=True)
OUT = f"results/moon_lazar_figure5_reps{REPS}.json"
with open(OUT, "w") as fh:
    json.dump({"reps": REPS, "settings": MOON_LAZAR_SETTINGS, "rows": rows},
              fh, indent=2)
print("\nwrote", OUT)

In [ ]:
import matplotlib.pyplot as plt

s = [r["sigma"] for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].errorbar(s, [r["pi_fpr"] for r in rows], yerr=[r["pi_fpr_se"] for r in rows],
               marker="o", label="ours")
ax[0].plot(s, [r["pi_fpr_published"] for r in rows], "s--", label="published")
ax[0].axhline(0.05, color="grey", lw=0.8, ls=":")
ax[0].set_title("Fig. 5a: false-positive rate")

ax[1].errorbar(s, [r["pi_power"] for r in rows], yerr=[r["pi_power_se"] for r in rows],
               marker="o", label="PI, ours")
ax[1].plot(s, [r["pi_power_published"] for r in rows], "s--", label="PI, published")
ax[1].errorbar(s, [r["pd_power"] for r in rows], yerr=[r["pd_power_se"] for r in rows],
               marker="^", label="PD, ours")
ax[1].plot(s, [r["pd_power_published"] for r in rows], "v--", label="PD, published")
ax[1].set_title("Fig. 5b: power")

for a in ax:
    a.set_xlabel("noise level sigma")
    a.set_ylim(-0.03, 1.03)
    a.legend(fontsize=8)
fig.tight_layout()
FIG = f"results/moon_lazar_figure5_reps{REPS}.png"
fig.savefig(FIG, dpi=150)
print("wrote", FIG)

In [ ]:
for output_file in (OUT, FIG):
    try:
        from google.colab import files
        files.download(output_file)
        print("Downloaded:", output_file)
    except Exception as e:
        print("(Not on Colab / download skipped):", e)